In [ ]:
# -*- coding: utf-8 -*-
"""PatchAntennaAI_R1_8_CNN (Real-magnitude only, CNN encoder)

Train a model that maps a **10×10 geometry** (100 bits) to real-valued **|S11| (magnitude)**
across **61 frequency points**, evaluate it, and plot **loss vs. epoch**.
"""

# 1) Setup & installs
!pip -q install gradio matplotlib pandas numpy

# 1a) Show environment info
import torch, pandas as pd, numpy as np, sys

print("Python version:", sys.version.split()[0])
print("PyTorch version:", torch.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

# 2) Configuration
from dataclasses import dataclass
@dataclass
class Cfg:
    csv_path: str = "/content/Full_1000Data.csv"
    test_csv_path: str = "/content/Full_1000Data.csv"
    input_dim: int = 100          # geometry bits
    seq_len: int = 61             # frequency points
    output_mode: str = "mag_only" # ONLY magnitude is supported now
    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 50
    val_split: float = 0.2
    dmodel: int = 128
    nhead: int = 8
    ffn_hidden: int = 256
    num_transformer_layers: int = 2
    num_spectral_blocks: int = 2
    top_k_freq: int = 16
    freq_hz_start: float = 1e9
    freq_hz_stop: float = 6e9
    device: str = "auto"
    transformer_dropout: float = 0.3
    weight_decay: float = 1e-5

    # --- Data augmentation (training data only, on geometry X) ---
    aug_noise_std: float = 0.01        # standard deviation for noise injection
    aug_freq_mask_prob: float = 0.5    # probability of applying a bit-mask per sample
    aug_freq_mask_width: int = 8       # max width (in bits) of the mask

CFG = Cfg(); CFG

# 2a) Display configuration
print("Current configuration:")
for k, v in CFG.__dict__.items():
    print(f"  {k}: {v}")

# 3) Imports & Dataset
import math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

def get_device(name: str):
    if name == "auto":
        return "cuda" if torch.cuda.is_available() else "cpu"
    return name

class AntennaDataset(Dataset):
    """
    Dataset for:
      - X: geometry bits [N, input_dim]
      - Y: real S11 magnitude [N, seq_len]

    CSV format (per row):
      [100 geometry bits] + [61 real magnitudes]
    """
    def __init__(self, csv_path, input_dim=100, seq_len=61, output_mode="mag_only"):
        if output_mode != "mag_only":
            raise ValueError("Only 'mag_only' mode is supported: CSV must be [100 geom + 61 real magnitudes].")
        df = pd.read_csv(csv_path)
        values = df.values.astype(np.float32)

        self.input_dim = input_dim
        self.seq_len = seq_len

        expected = input_dim + seq_len
        if values.shape[1] < expected:
            raise ValueError(
                f"CSV has {values.shape[1]} columns; needs ≥ {expected} "
                f"(100 geometry + 61 real magnitudes)."
            )

        x = values[:, :input_dim]
        y = values[:, input_dim:input_dim+seq_len]  # real magnitudes only

        self.X = torch.from_numpy(x)        # [N, 100]
        self.Y = torch.from_numpy(y)        # [N, 61]

    def __len__(self):
        return self.X.size(0)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# 3a) Inspect dataset shapes and a sample
try:
    ds_debug = AntennaDataset(
        CFG.csv_path,
        input_dim=CFG.input_dim,
        seq_len=CFG.seq_len,
        output_mode=CFG.output_mode
    )
    print("Dataset successfully loaded.")
    print("  Number of samples:", len(ds_debug))

    x0, y0 = ds_debug[0]
    print("  Input sample shape:", x0.shape)      # expected: [100]
    print("  Target sample shape:", y0.shape)     # expected: [61]

    print("  First 10 input bits:", x0[:10].tolist())
    print("  First 5 frequency magnitudes:", y0[:5].tolist())
except Exception as e:
    print("Error loading dataset:", e)

# 4) Model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        # x: [B, L, D]
        L = x.size(1)
        return x + self.pe[:, :L, :]

class SpectralBlock(nn.Module):
    def __init__(self, d_model, seq_len, top_k=16, ffn_hidden=256):
        super().__init__()
        self.top_k = min(top_k, seq_len // 2 + 1)
        self.w_real = nn.Parameter(torch.randn(d_model, self.top_k) * 0.02)
        self.w_imag = nn.Parameter(torch.randn(d_model, self.top_k) * 0.02)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_hidden),
            nn.GELU(),
            nn.Linear(ffn_hidden, d_model)
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: [B, L, D]
        residual = x
        B, L, D = x.shape
        x = self.ln1(x)

        # [B, D, L]
        x_ch = x.transpose(1, 2)
        X = torch.fft.rfft(x_ch, dim=-1)  # [B, D, L_freq]

        k = self.top_k
        idx = torch.arange(k, device=X.device)
        Xk = X[..., idx]                  # [B, D, k]

        a, b = Xk.real, Xk.imag
        wr, wi = self.w_real.unsqueeze(0), self.w_imag.unsqueeze(0)

        real = a*wr - b*wi
        imag = a*wi + b*wr
        Xk_mod = torch.complex(real, imag)

        X_new = torch.zeros_like(X)
        X_new[..., idx] = Xk_mod

        x_time = torch.fft.irfft(X_new, n=L, dim=-1).transpose(1, 2)  # [B, L, D]

        x = residual + x_time
        y = self.ff(self.ln2(x))
        return x + y

class FEDformerDecoder(nn.Module):
    """
    FEDformer-style decoder that outputs a single real channel per frequency:
      output: [B, seq_len]
    """
    def __init__(self, d_model, seq_len, nhead=8, ffn_hidden=256,
                 num_transformer_layers=2, num_spectral_blocks=2, top_k=16, dropout=0.1):
        super().__init__()
        self.pos = PositionalEncoding(d_model, max_len=seq_len)
        self.spectral = nn.ModuleList([
            SpectralBlock(d_model, seq_len, top_k=top_k, ffn_hidden=ffn_hidden)
            for _ in range(num_spectral_blocks)
        ])

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ffn_hidden,
            batch_first=True,
            activation="gelu",
            norm_first=True,
            dropout=dropout
        )
        self.tr = nn.TransformerEncoder(enc_layer, num_layers=num_transformer_layers)
        self.head = nn.Linear(d_model, 1)   # single real channel

    def forward(self, tokens):
        # tokens: [B, L, D]
        z = self.pos(tokens)
        for blk in self.spectral:
            z = blk(z)
        z = self.tr(z)                # [B, L, D]
        out = self.head(z)            # [B, L, 1]
        return out.squeeze(-1)        # [B, L]

class CNNEncoder(nn.Module):
    """
    CNN-based encoder for 10×10 binary geometry instead of a Graph Neural Network.
    Architecture:
        Conv2d(1 → 16, kernel=3, padding=1)
        Conv2d(16 → 32, kernel=3, padding=1)
        Conv2d(32 → 64, kernel=3, padding=1)
        Flatten
        Linear → d_model
    """
    def __init__(self, d_model=128):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # After convs: [B, 64, 10, 10] → flatten to [B, 6400]
        self.fc = nn.Linear(64 * 10 * 10, d_model)

    def forward(self, geom_bits):
        # geom_bits: [B, 100] → reshape to [B, 1, 10, 10]
        x = geom_bits.view(-1, 1, 10, 10)
        h = self.conv(x)
        h = h.flatten(start_dim=1)         # [B, 6400]
        g = self.fc(h)                     # [B, d_model]
        return g

class Geometry2SParam(nn.Module):
    """
    Full model:
      geometry [B, 100] -> CNN encoder -> tokens -> FEDformer decoder -> [B, 61]
    """
    def __init__(self, seq_len=61, dmodel=128, nhead=8,
                 ffn_hidden=256, num_transformer_layers=2,
                 num_spectral_blocks=2, top_k=16, dropout=0.1):
        super().__init__()

        # CNN-based encoder
        self.encoder = CNNEncoder(d_model=dmodel)

        # Map encoder output → transformer tokens
        self.to_tokens = nn.Linear(dmodel, seq_len * dmodel)

        self.decoder = FEDformerDecoder(
            dmodel, seq_len, nhead,
            ffn_hidden, num_transformer_layers,
            num_spectral_blocks, top_k,
            dropout=dropout
        )
        self.seq_len, self.dmodel = seq_len, dmodel

    def forward(self, geom_bits):
        g = self.encoder(geom_bits)                          # [B, dmodel]
        tokens = self.to_tokens(g).view(-1, self.seq_len, self.dmodel)  # [B, L, D]
        return self.decoder(tokens)                          # [B, 61]

def mag_mse(pred, target):
    """
    MSE on real-valued magnitudes:
      pred, target: [B, seq_len]
    """
    return F.mse_loss(pred, target)

# 4a) Build model, show summary, and test a forward pass
device = get_device(CFG.device)
print("Using device:", device)

model_dbg = Geometry2SParam(
    seq_len=CFG.seq_len,
    dmodel=CFG.dmodel,
    nhead=CFG.nhead,
    ffn_hidden=CFG.ffn_hidden,
    num_transformer_layers=CFG.num_transformer_layers,
    num_spectral_blocks=CFG.num_spectral_blocks,
    top_k=CFG.top_k_freq,
    dropout=CFG.transformer_dropout
).to(device)

print("\nModel architecture:\n")
print(model_dbg)

total_params = sum(p.numel() for p in model_dbg.parameters())
trainable_params = sum(p.numel() for p in model_dbg.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Dummy forward pass
dummy_x = torch.randint(0, 2, (2, CFG.input_dim), dtype=torch.float32).to(device)
with torch.no_grad():
    dummy_y = model_dbg(dummy_x)   # [B, 61]

print("\nDummy input shape:", dummy_x.shape)   # [B, 100]
print("Dummy output shape:", dummy_y.shape)    # [B, 61]

# 4.5) Data augmentation utilities (training data only)

def augment_batch(x, y, cfg: Cfg):
    """
    Apply data augmentation to training batch on the *input geometry*:
      1) Noise injection on geometry bits
      2) Random contiguous bit masking

    x: [B, input_dim]       (augmented)
    y: [B, seq_len]         (targets unchanged)
    """
    # Work on a copy so original batch is untouched if needed
    x = x.clone()

    B, D = x.shape  # D = input_dim (100)

    # ---- 1) Noise Injection on geometry bits ----
    if cfg.aug_noise_std > 0.0:
        noise = torch.randn(B, D, device=x.device) * cfg.aug_noise_std
        x = x + noise

    # ---- 2) Random contiguous bit masking ----
    if cfg.aug_freq_mask_prob > 0.0 and cfg.aug_freq_mask_width > 0:
        max_width = min(cfg.aug_freq_mask_width, D)
        for b in range(B):
            if torch.rand(1, device=x.device).item() < cfg.aug_freq_mask_prob:
                w = torch.randint(1, max_width + 1, (1,), device=x.device).item()
                start = torch.randint(0, D - w + 1, (1,), device=x.device).item()
                x[b, start:start + w] = 0.0

    return x, y

# 4.6a) Test data augmentation on a single batch
if 'ds_debug' not in globals():
    ds_debug = AntennaDataset(
        CFG.csv_path,
        input_dim=CFG.input_dim,
        seq_len=CFG.seq_len,
        output_mode=CFG.output_mode
    )

loader_debug = DataLoader(ds_debug, batch_size=4, shuffle=True)
xb0, yb0 = next(iter(loader_debug))
print("Original batch shapes:", xb0.shape, yb0.shape)

xb_aug, yb_aug = augment_batch(xb0, yb0, CFG)
print("Augmented batch shapes:", xb_aug.shape, yb_aug.shape)

print("\nFirst sample, first 10 geometry bits BEFORE aug:", xb0[0, :10])
print("First sample, first 10 geometry bits AFTER  aug:", xb_aug[0, :10])

# 4.7) Display the FULL augmented training dataset (offline augmentation preview)

def build_full_augmented_dataset(cfg: Cfg):
    """
    Return:
        X_aug_all: tensor [N_train, 100]    (AUGMENTED geometry)
        Y_aug_all: tensor [N_train, 61]     (original real-magnitude targets)

    This generates *one pass* of augmented input data across the entire training split.
    NOTE: Because augmentation includes randomness, each run gives different results.
    """
    print("Building full augmented dataset preview...")

    ds = AntennaDataset(
        cfg.csv_path,
        input_dim=cfg.input_dim,
        seq_len=cfg.seq_len,
        output_mode=cfg.output_mode
    )

    n_total = len(ds)
    n_val = int(cfg.val_split * n_total)
    n_train = n_total - n_val

    train_ds, _ = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=False)

    X_aug_list = []
    Y_aug_list = []

    for xb, yb in train_loader:
        xb_aug, yb_aug = augment_batch(xb, yb, cfg)  # xb_aug changed, yb_aug == yb
        X_aug_list.append(xb_aug)
        Y_aug_list.append(yb_aug)

    X_aug_all = torch.cat(X_aug_list, dim=0)
    Y_aug_all = torch.cat(Y_aug_list, dim=0)

    print("Augmented dataset built successfully!")
    print("  Augmented X shape:", X_aug_all.shape)  # [N_train, 100]
    print("  Y shape (targets, unchanged):", Y_aug_all.shape)  # [N_train, 61]

    return X_aug_all, Y_aug_all


# ---- Run it to preview augmented dataset ----
X_aug_full, Y_aug_full = build_full_augmented_dataset(CFG)

print("\nExample: First 10 geometry bits of sample 0:")
print(X_aug_full[0, :10])

print("\nExample: First 5 augmented freq magnitudes of sample 0:")
print(Y_aug_full[0, :5])

import os, csv, matplotlib.pyplot as plt

hist_epoch, hist_train, hist_val = [], [], []
history_csv_path = "/content/history.csv"

def train_model(cfg: Cfg, weights_path: str = "/content/gnn_fedformer_cnn_best.pt"):
    device = get_device(cfg.device)
    print("Starting training on device:", device)

    ds = AntennaDataset(
        cfg.csv_path,
        input_dim=cfg.input_dim,
        seq_len=cfg.seq_len,
        output_mode=cfg.output_mode
    )

    n_total = len(ds)
    n_val = int(cfg.val_split * n_total)
    n_train = n_total - n_val
    print(f"Total samples: {n_total} | Train: {n_train} | Val: {n_val}")

    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

    # Reset history file
    with open(history_csv_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["epoch", "train_loss", "val_loss"])

    model = Geometry2SParam(
        seq_len=cfg.seq_len,
        dmodel=cfg.dmodel,
        nhead=cfg.nhead,
        ffn_hidden=cfg.ffn_hidden,
        num_transformer_layers=cfg.num_transformer_layers,
        num_spectral_blocks=cfg.num_spectral_blocks,
        top_k=cfg.top_k_freq,
        dropout=cfg.transformer_dropout
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_val = float("inf")
    patience, left = 10, 10

    for epoch in range(1, cfg.epochs+1):
        # -------- TRAIN --------
        model.train()
        tr_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)  # xb: [B, 100], yb: [B, 61]

            # Data augmentation on geometry only
            xb_aug, yb = augment_batch(xb, yb, cfg)

            opt.zero_grad()
            yhat = model(xb_aug)        # [B, 61]
            loss = mag_mse(yhat, yb)
            loss.backward()
            opt.step()
            tr_loss += loss.item() * xb.size(0)

        tr_loss /= max(1, n_train)

        # -------- VALIDATION (NO AUGMENTATION) --------
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                yhat = model(xb)        # [B, 61]
                loss = mag_mse(yhat, yb)
                val_loss += loss.item() * xb.size(0)
        val_loss /= max(1, n_val)

        hist_epoch.append(epoch)
        hist_train.append(tr_loss)
        hist_val.append(val_loss)
        with open(history_csv_path, "a", newline="") as f:
            w = csv.writer(f)
            w.writerow([epoch, tr_loss, val_loss])

        print(f"Epoch {epoch:03d} | train {tr_loss:.6f} | val {val_loss:.6f}")
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            torch.save(model.state_dict(), weights_path)
            print("  ↳ saved:", weights_path)
            left = patience
        else:
            left -= 1
            if left <= 0:
                print("Early stopping.")
                break

    print("\nTraining complete.")
    print("Best validation loss recorded:", best_val)
    print("Weights saved to:", weights_path)

    return weights_path, best_val

# 6) Plot loss vs epoch
def plot_loss_curve():
    plt.figure()
    plt.plot(hist_epoch, hist_train, label="Train")
    plt.plot(hist_epoch, hist_val,   label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE on |S11|)")
    plt.title("Training / Validation Loss vs. Epoch (Real Magnitude Only, CNN Encoder)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# 6a) Show first few history entries
from itertools import islice

print("First few logged epochs / losses:")
for e, tr, va in islice(zip(hist_epoch, hist_train, hist_val), 0, 5):
    print(f"  Epoch {e:03d}: train={tr:.6f}, val={va:.6f}")

# 7) Train
weights_path, best_val = train_model(CFG, weights_path="/content/gnn_fedformer_cnn_best.pt")
print("Best validation loss:", best_val)

print("\nTraining finished.")
print("Best model path:", weights_path)
print("Best validation loss:", best_val)

# 8) Plot loss curve
plot_loss_curve()

# 8a) Check saved model file
if os.path.exists(weights_path):
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Model file found: {weights_path} ({size_mb:.2f} MB)")
else:
    print("Model weights file not found:", weights_path)
